In [17]:
# possible hidden states in the HMM (E = exon, 5 = donor splice site, I = intron)
s = ['E', '5', 'I']

# initial probability distribution over states (we start in state E)
sp = {'E': 1.0, '5': 0.0, 'I': 0.0}

# transition probabilities between states
tp = {
    'E': {'E': 0.9, '5': 0.1, 'I': 0.0},
    '5': {'I': 1.0, 'E': 0.0, '5': 0.0},
    'I': {'I': 0.9, 'E': 0.1, '5': 0.0},
}

# emission probabilities (for each nucleotide from each state)
ep = {
    'E': dict.fromkeys(['A','C','G','T'], 0.25),  # Equal chance for any base
    '5': {'A': 0.0, 'C': 0.0, 'G': 1.0, 'T': 0.0}, # Only G is allowed at donor site
    'I': {'A': 0.4, 'C': 0.1, 'G': 0.1, 'T': 0.4},
}


In [18]:
import math

# computing log probability of an observed sequence given a specific path

def calc_log_prob(pth, observed, sp = sp , tp = tp , ep = ep ):
    if len(pth) != len(observed):
        raise ValueError("pth and observed sequence must have the same length")

    logp = 0.0
    s0 = pth[0]
    logp += math.log(sp[s0])
    logp += math.log(ep[s0][observed[0]])

    for i in range(1, len(pth)):
        prev_st = pth[i-1]
        curr_st = pth[i]
        logp += math.log(tp[prev_st][curr_st])
        logp += math.log(ep[curr_st][observed[i]])

    return round(logp, 2)

# Example usage
given_pth = "EEEEEEEEEEEEEEE5IIIIIIIIII"
observed_seq  = "CTTCATGTGAAAGCAGACGTAAGTCA"
print("Log probability of given pth:", calc_log_prob(given_pth, observed_seq))

Log probability of given pth: -40.23


In [19]:
# implementing the viterbi algo - finds the most probable path

# Observation sequence
obs = list("CTTCATGTGAAAGCAGACGTAAGTCA")

# Viterbi algorithm
vt = [{}]
bp = [{}]

for st in s:
    vt[0][st] = sp[st] * ep[st].get(obs[0], 0)
    bp[0][st] = None

for t in range(1, len(obs)):
    vt.append({})
    bp.append({})
    for cur in s:
        mp, prev_st = max(
            (vt[t-1][pre] * tp[pre][cur] * ep[cur].get(obs[t], 0), pre) for pre in s
        )
        vt[t][cur] = mp
        bp[t][cur] = prev_st

# Backtrack to find best path
fp = []
last_st = max(vt[-1], key=vt[-1].get)
fp.append(last_st)

for t in range(len(obs)-1, 0, -1):
    last_st = bp[t][last_st]
    fp.insert(0, last_st)

print("".join(fp))


EEEEEEEEEEEEEEEEEEEEEEEEEE
